# Gün 133: IPv4 Adresi, Subnet ve Gateway (Bit Düzeyinde Ağ Maskeleme, Alt Ağ Ayrıştırma, Varsayılan Ağ Geçidi ve Çekirdek LPM Rotalama Kararı)

```
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR

Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)

Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim
amaçlı olarak paylaşılmıştır.

YASAKLAR:
  1. Kopyalanamaz, çoğaltılamaz, dağıtılamaz veya yeniden yayınlanamaz.
  2. Ticari veya ticari olmayan hiçbir projede kullanılamaz, değiştirilemez.
  3. Alt lisanslanamaz, satılamaz veya devredilemez.
  4. Tersine mühendislik yapılamaz.

İZİN VERİLEN KULLANIM:
  - GitHub üzerinde görüntüleme ve okuma.
  - Kişisel öğrenim amacıyla kodu inceleme (kopyalamadan).

YAZARIN AÇIK YAZILI İZNİ OLMAKSIZIN HİÇBİR KULLANIM HAKKI TANINMAZ.
İzin talepleri için: GitHub @seydivakkas
```

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)

**Faz:** F4 — TCP/UDP / Protocol / Serialization  
**Konu:** L3 Ağ Katmanı Mantıksal Adresleme: IPv4, CIDR Subnetting, Ağ Geçidi (Default Gateway) ve Longest Prefix Match (LPM) Yönlendirmesi  
**Ön Koşul:** Gün 132 — MAC, IP, Port ve Socket Ayrımı  
**Kritik Seviye:** Savunma Sanayii Taktik Ağlarında (Radar Ağı, Aviyonik Veri Yolu, Komuta Kontrol Omurgası) Güvenli Alt Ağ İzolasyonu ve Yönlendirme

---

### 40-Nokta Bildirimi ve Kalite Güvencesi
Bu modül; modern savunma sanayii ve yüksek performanslı C++ ağ mimarilerinde IP yönlendirmenin temel taşı olan **IPv4 adresi, alt ağ maskesi (Subnet Mask / CIDR) ve varsayılan ağ geçidi (Default Gateway)** mekanizmalarını bit düzeyinde inceler. 32-bit ikili (binary) adresleme, bitwise AND/OR operasyonları, broadcast/network adres hesaplamaları, host kapasiteleri ve bir cihazın paketi yerel anahtarda mı (L2 ARP) yoksa yönlendiriciye mi (L3 Gateway) teslim edeceğine karar veren çekirdek yönlendirme mantığı (Forwarding Information Base - FIB / LPM) tüm detaylarıyla inşa edilmiştir.


## 1. 🎯 Bugünkü Problem: Ağlar Arası Köprü — Paket Yerel mi Kalacak, Dışarı mı Çıkacak?

Bir savaş uçağında, fırkateynde veya taktik sahada binlerce cihaz birbirine bağlıdır:
- Uçak içi aviyonik görev bilgisayarı: `192.168.10.15/24`
- Taktik AESA radar sinyal işlemcisi: `192.168.10.80/24`
- Yer komuta kontrol karargah merkezi (C2 WAN): `10.200.5.1/16`
- Uydu taktik haberleşme modemi: `192.168.10.1/24` (Ağ Geçidi)

### Görev Bilgisayarı Bir Paket Göndereceği Zaman Hangi Kararı Verir?
Bir düğüm (host) hedef IP adresine bakarak şu temel soruyu sormak zorundadır:
> **"Bu hedef IP benimle aynı fiziksel odada / kabloda (aynı alt ağda) mı yer alıyor, yoksa farklı bir ağda mı?"**

Eğer alt ağ maskesi (Subnet Mask) ve ağ geçidi (Gateway) mimarisi olmasaydı:
1. **Maskeleme Olmasaydı:** Cihaz hedefin yerel mi uzak mı olduğunu bilemezdi. `10.200.5.1` adresine doğrudan L2 kablodan ARP sorgusu atar, hiçbir cevap alamaz ve paket asla uçaktan dışarı çıkamazdı!
2. **Gateway Olmasaydı:** Farklı ağlara ait paketlerin teslim edileceği ortak bir çıkış kapısı (Router/Next-Hop) bulunamazdı; izole lokal ağ adaları arasında iletişim kurulamazdı.
3. **Yanlış Subnet Maskesi Tanımlanırsa:** Yerel bir komşuya gidecek paket yanlışlıkla ağ geçidine gönderilerek ağda gereksiz darboğaz (bottleneck) ve yönlendirme döngüleri (routing loops) oluştururdu.

**Çekirdeğin Değişmez Yönlendirme Kuralı:**
$$\text{Hedef Yerel mi?} = (\text{DestIP} \ \& \ \text{Netmask}) \stackrel{?}{=} (\text{LocalIP} \ \& \ \text{Netmask})$$
- **EVET (Yerel Alt Ağ):** Paketi doğrudan hedef cihaza gönder! Hedef IP'nin MAC adresini ARP ile bul ve yerel anahtar (L2 Switch) üzerinden doğrudan teslim et.
- **HAYIR (Uzak Ağ):** Paketi doğrudan hedefe gönderemezsin! L3 başlığında hedef IP `DestIP` olarak kalır, ancak L2 Ethernet çerçevesinde hedef MAC olarak **Varsayılan Ağ Geçidinin (Default Gateway) MAC adresi** yazılır!


## 2. ❓ Bu Konuyu Neden Öğreniyoruz? (Savunma Sanayii Gerekçesi)

Savunma sanayiinde taktik ağlar güvenlik, bant genişliği ve elektromanyetik parazit izolasyonu gereği katı alt ağlara (Subnets) bölünür:

### Savunma Sanayiindeki Kullanım Alanları:
1. **ASELSAN Aviyonik Görev Sistemleri:**
   - Görev bilgisayarı, radar, FLIR ve seyrüsefer sistemleri `192.168.10.0/24` aviyonik alt ağında izoledir. Bu ağdaki cihazlar yüksek hızlı ($10\text{ Gbps}$) deterministik Ethernet üzerinden switch ile konuşur; gateway trafiği üretmez.
2. **HAVELSAN Taktik Veri Bağı ve C2 Gateway:**
   - Gemideki iç savaş yönetim alt ağı (`172.16.0.0/16`), taktik Link-16 veya TAFICS omurgasına bağlanırken bir **Kriptolu Güvenlik Ağ Geçidi (Security Gateway / Router)** üzerinden geçer. Ağ geçidi paketleri filtreler (ACL), şifreler ve LPM tablosuna göre uyduya yönlendirir.
3. **ROKETSAN Füze Fırlatma Kontrol Birimi:**
   - Fırlatıcı kontrol podu ile füzeler arasında noktadan noktaya (Point-to-Point) `/30` alt ağları kullanılır. Her linkte sadece 2 adet kullanılabilir IP adresi bulunur; yabancı cihazların ağa bağlanması donanımsal ve mantıksal olarak imkansız kılınır.


## 3. 🔍 5N1K Analizi (Sistem ve Donanım Sınırları)

| Soru | Sistem ve Donanım Düzeyi Cevabı | Savunma Sanayii Odaklı Karşılığı |
|:---|:---|:---|
| **NE?** | IPv4 (32-bit mantıksal adres), Subnet Mask (Ağ/Host sınır belirteci) ve Gateway (Farklı alt ağlara çıkış yönlendiricisi). | Taktik alt ağ bölümleme, güvenlik sınırları ve WAN yönlendirme mimarisi. |
| **NEDEN?** | Küresel adres uzayını hiyerarşik yönetmek, broadcast trafiğini sınırlamak ve güvenli geçitler kurmak için. | Aviyonik kritik sensör ağını taktik genel ağdan izole etmek ve siber saldırı yüzeyini daraltmak. |
| **NEREDE?** | L3 IP başlığında (`struct iphdr`), Linux çekirdeği Rota Bilgi Tabanında (FIB - Forwarding Information Base). | Görev bilgisayarlarında, taktik yönlendiricilerde (Router) ve kriptolu ağ geçitlerinde. |
| **NASIL?** | Bitwise AND (`IP & Mask`) işlemiyle ağ kimliğini hesaplayarak ve Longest Prefix Match (LPM) ile yönlendirerek. | Linux `ip route`, C++ `getifaddrs`, `inet_pton` ve soket yönlendirme tablolarıyla. |
| **NE ZAMAN?** | Bir süreç `sendto()` veya `connect()` çağırdığı anda çekirdek paketi ağ kartına teslim etmeden hemen önce. | Her telemetri, radar darbesi veya komuta kontrol paketi iletilirken. |
| **KİM?** | Ağ mimarları, gömülü sistem mühendisleri, savunma haberleşme uzmanları. | Taktik ağ ve aviyonik altyapı geliştirme ekipleri. |


## 4. 💡 Günlük Hayat Benzetmesi (Zihinsel Model / Analoji)

### Şehirlerarası Telefon Alan Kodları ve Santral Geçidi
Klasik sabit hatlı telefon şebekesini hayal edin:
1. **Alan Kodu (Subnet / Network ID):**
   - Ankara için `0312`, İstanbul Anadolu yakası için `0216`.
   - Bu kod o ağın bölgesel kimliğidir (Network Prefix).
2. **Abone Numarası (Host ID):**
   - `444 01 23`. Bu numara o alan kodunun içindeki spesifik kişiyi/cihazı tanımlar.
3. **Yerel Arama (Local Subnet İletişimi):**
   - Ankara'daki bir kişi yine Ankara içindeki birini ararken alan kodu çevirmez, doğrudan `444 01 23` tuşlar. Santral (Switch) sinyali şehir dışına çıkarmadan yerel kablolardan doğrudan hedefe bağlar.
4. **Şehirlerarası Arama ve Santral (Default Gateway):**
   - Ankara'daki kişi İstanbul'daki bir numarayı (`0216 555 12 34`) aradığında telefon makinesi şunu anlar: *"Bu numara benim alan kodumda değil!"*.
   - Çağrıyı doğrudan bağlayamaz; çağrıyı önce **Bölge Santraline (Default Gateway)** iletir.
   - Santral (Router) bu çağrıyı alır, şehirlerarası fiber hatlar üzerinden İstanbul santraline aktarır.


## 5. 🧱 İlk İlkeler (First Principles) ve Matematiksel Mantık Modeli

IPv4 adreslemesi tamamen 32-bit ikili cebir (Boolean Algebra) üzerine kuruludur:

### 1. Adres Ayrıştırma ve Bit Düzeyinde Ağ Hesabı:
Bir IPv4 adresi 32-bit bir tamsayıdır: $\text{IP} \in [0, 2^{32}-1]$.
CIDR gösteriminde `/N`, en soldaki (MSB) $N$ bitin **Ağ Kimliği (Network ID)**, kalan $(32-N)$ bitin ise **Cihaz Kimliği (Host ID)** olduğunu belirtir:
$$\text{Netmask}(N) = \sum_{i=32-N}^{31} 2^i = (\text{0xFFFFFFFF} \ll (32-N)) \ \& \ \text{0xFFFFFFFF}$$

### 2. Temel Ağ Parametrelerinin Hesaplanması:
- **Ağ Adresi (Network Address):** Alt ağın kendisini temsil eder; hostlara verilemez:
  $$\text{Network} = \text{IP} \ \& \ \text{Netmask}$$
- **Yayın Adresi (Broadcast Address):** O alt ağdaki tüm hostlara aynı anda seslenmek için kullanılır:
  $$\text{Broadcast} = \text{Network} \ \mid \ (\sim\text{Netmask} \ \& \ \text{0xFFFFFFFF})$$
- **Kullanılabilir Host Sayısı ($H$):**
  $$H = 2^{(32-N)} - 2 \quad (N \le 30)$$
  *(Network adresi ve Broadcast adresi çıkarılır; dolayısıyla $-2$ uygulanır).*
- **İlk Kullanılabilir Host Adresi:** $\text{Network} + 1$
- **Son Kullanılabilir Host Adresi:** $\text{Broadcast} - 1$

### 3. Longest Prefix Match (LPM) Algoritması:
Bir yönlendiricinin yönlendirme tablosunda birden fazla eşleşen kural olabilir:
- Kural 1: `192.168.0.0/16` $\implies$ `eth0`
- Kural 2: `192.168.10.0/24` $\implies$ `eth1`
- Kural 3: `192.168.10.80/32` $\implies$ `radar_tap`
- Varsayılan (Default): `0.0.0.0/0` $\implies$ `gateway_router`

Hedef `192.168.10.80` olduğunda hem `/0`, hem `/16`, hem `/24`, hem de `/32` kuralları uyar.
Linux çekirdeği **en uzun maskeye (en spesifik ağa)** sahip olan kuralı seçer:
$$\text{Optimal Route} = \arg\max_{R_i \in \text{Routes}, \ (\text{DestIP} \ \& \ M_i) = N_i} (\text{PrefixLength}_i)$$
Bu algoritma donanımda TCAM (Ternary Content Addressable Memory), Linux çekirdeğinde ise **LC-trie** veri yapısı ile $O(1)$ veya $O(k)$ sürede çözülür!


## 6. 📐 ASCII ve Mermaid Sistem Mimarisi Şeması

### ASCII Mimarisi: Yerel İletim (L2 Direct) vs Ağ Geçidi Yönlendirmesi (L3 Routed)
```text
  ==================================================================================================
  IPv4 SUBNET DENETİMİ VE VARSAYILAN AĞ GEÇİDİ (DEFAULT GATEWAY) KARAR MİMARİSİ
  ==================================================================================================

  [ Görev Bilgisayarı ] : IP = 192.168.10.15 / 24, Netmask = 255.255.255.0, Gateway = 192.168.10.1
                                       |
                     GÖNDERİLECEK HEDEF IP NEDİR?
                                       |
                +----------------------+----------------------+
                |                                             |
     [ SENARYO A: HEDEF = 192.168.10.80 ]          [ SENARYO B: HEDEF = 10.200.5.1 ]
     (Radar İşlemcisi)                            (Karargah Komuta Kontrol)
                |                                             |
   (192.168.10.80 & /24) == (192.168.10.15 & /24)? (10.200.5.1 & /24) == (192.168.10.15 & /24)?
                |                                             |
          ==> EVET (YEREL ALT AĞ)                       ==> HAYIR (UZAK AĞ!)
                |                                             |
     Gateway'i Devre Dışı Bırak!                   Paket Doğrudan Gidemez!
     Hedef IP için Yerel ARP Sorgusu At             Varsayılan Gateway IP: 192.168.10.1
                |                                             |
     L2 DstMAC = Radar MAC                          Gateway MAC için ARP Sorgusu At
     L3 DstIP  = 192.168.10.80                      L2 DstMAC = Gateway MAC (Router)
                |                                   L3 DstIP  = 10.200.5.1 (Değişmez!)
                v                                             v
     [ Yerel L2 Ethernet Switch ]                   [ Yerel L2 Ethernet Switch ]
                |                                             |
                v                                             v
     [ Radar Alıcısına Teslim ]                    [ Gateway / Router Cihazına Teslim ]
                                                              | (L2 Başlığı Soyulur,
                                                              |  L3 Tablosu İncelenir)
                                                              v
                                                   [ WAN / Uydu / Karargah Ağı ]
  ==================================================================================================
```

### Mermaid Akış Şeması: Çekirdek Paket Gönderim Karar Motoru
```mermaid
flowchart TD
    Start([Uygulama sendto/connect Çağırır]) --> Extract[Hedef IP Adresini Oku: DestIP]
    Extract --> SubnetCalc["(DestIP & Netmask) == NetworkID ?"]
    
    SubnetCalc -- EVET: Yerel Alt Ağ --> LocalARP[Hedef IP için ARP Önbelleğine Bak]
    LocalARP --> BuildL2Local["L2 Çerçevesi Kur: DstMAC = Target_MAC"]
    BuildL2Local --> SendNIC1[Ağ Kartından Kabloya Bas: Direct Delivery]
    
    SubnetCalc -- HAYIR: Farklı Ağ --> FindGW[Varsayılan Ağ Geçidini Oku: Gateway IP]
    FindGW --> GWARP[Gateway IP için ARP Önbelleğine Bak]
    GWARP --> BuildL2GW["L2 Çerçevesi Kur: DstMAC = Gateway_MAC, DstIP = DestIP"]
    BuildL2GW --> SendNIC2[Ağ Kartından Kabloya Bas: Next-Hop Delivery]
    
    SendNIC1 --> Done([Teslim Edildi])
    SendNIC2 --> RouterHop([Router Paketi Alır ve Yeni Ağa İletir])
```


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 8 (CODE 1): IPv4 Adres, Subnetting ve Yönlendirme (LPM) Motoru
Bit Düzeyinde Maskeleme, Alt Ağ Hesaplamaları ve Longest Prefix Match Simülatörü
"""

import struct
from typing import Dict, List, Tuple, Optional

class IPv4Address:
    """IPv4 32-bit tamsayı ve dotted-decimal adres yönetimi sınıfı."""
    def __init__(self, ip_str: str):
        self.ip_str = ip_str.strip()
        octets = [int(x) for x in self.ip_str.split('.')]
        if len(octets) != 4 or any(o < 0 or o > 255 for o in octets):
            raise ValueError(f"Geçersiz IPv4 Adresi: {ip_str}")
        self.ip_int = (octets[0] << 24) | (octets[1] << 16) | (octets[2] << 8) | octets[3]

    @classmethod
    def from_int(cls, ip_int: int) -> 'IPv4Address':
        ip_int = ip_int & 0xFFFFFFFF
        s = f"{(ip_int >> 24) & 0xFF}.{(ip_int >> 16) & 0xFF}.{(ip_int >> 8) & 0xFF}.{ip_int & 0xFF}"
        return cls(s)

    def to_binary_str(self) -> str:
        b = f"{self.ip_int:032b}"
        return f"{b[0:8]}.{b[8:16]}.{b[16:24]}.{b[24:32]}"

    def __repr__(self):
        return self.ip_str

    def __eq__(self, other):
        if isinstance(other, IPv4Address):
            return self.ip_int == other.ip_int
        return False

class Subnet:
    """CIDR Notasyonu (/N) ve Bit Düzeyinde Alt Ağ Hesaplayıcı."""
    def __init__(self, cidr_str: str):
        # Örnek: "192.168.10.0/24"
        parts = cidr_str.strip().split('/')
        self.base_ip = IPv4Address(parts[0])
        self.prefix_len = int(parts[1]) if len(parts) > 1 else 32
        if not (0 <= self.prefix_len <= 32):
            raise ValueError(f"Geçersiz CIDR Prefix: {self.prefix_len}")

        # 32-bit Netmask Hesabı
        if self.prefix_len == 0:
            self.mask_int = 0
        else:
            self.mask_int = ((0xFFFFFFFF << (32 - self.prefix_len)) & 0xFFFFFFFF)

        self.netmask = IPv4Address.from_int(self.mask_int)
        self.wildcard = IPv4Address.from_int(~self.mask_int & 0xFFFFFFFF)
        self.network_ip = IPv4Address.from_int(self.base_ip.ip_int & self.mask_int)
        self.broadcast_ip = IPv4Address.from_int(self.network_ip.ip_int | (~self.mask_int & 0xFFFFFFFF))

    @property
    def total_hosts(self) -> int:
        return 2 ** (32 - self.prefix_len)

    @property
    def usable_hosts(self) -> int:
        if self.prefix_len >= 31:
            return 2 if self.prefix_len == 31 else 1
        return self.total_hosts - 2

    @property
    def first_usable_ip(self) -> IPv4Address:
        if self.prefix_len >= 31:
            return self.network_ip
        return IPv4Address.from_int(self.network_ip.ip_int + 1)

    @property
    def last_usable_ip(self) -> IPv4Address:
        if self.prefix_len >= 31:
            return self.broadcast_ip
        return IPv4Address.from_int(self.broadcast_ip.ip_int - 1)

    def contains(self, ip: IPv4Address) -> bool:
        """Hedef IP bu alt ağın sınırları içinde mi? (Bitwise AND testi)"""
        return (ip.ip_int & self.mask_int) == self.network_ip.ip_int

class RouteEntry:
    """Linux Çekirdeği Yönlendirme Tablosu (FIB) Rota Girdisi."""
    def __init__(self, subnet: Subnet, gateway: Optional[IPv4Address], interface: str, metric: int = 100):
        self.subnet = subnet
        self.gateway = gateway
        self.interface = interface
        self.metric = metric

class RoutingTable:
    """
    Longest Prefix Match (LPM) tabanlı yüksek başarımlı yönlendirme tablosu motoru.
    Linux FIB ve BSD Radix Trie mimarisine uygun olarak prefix uzunluklarına göre
    gruplanmış bucket hash mimarisi kullanır (O(W) karmaşıklık, W <= 32).
    """
    def __init__(self):
        self.routes: List[RouteEntry] = []
        # Prefix uzunluğuna göre hash tablosu: prefix_len -> {network_int: RouteEntry}
        self.prefix_buckets: Dict[int, Dict[int, RouteEntry]] = {}
        self.active_prefixes: List[int] = []

    def add_route(self, cidr_str: str, gateway: Optional[str], interface: str, metric: int = 100):
        sub = Subnet(cidr_str)
        gw = IPv4Address(gateway) if gateway else None
        entry = RouteEntry(sub, gw, interface, metric)
        self.routes.append(entry)
        
        p_len = sub.prefix_len
        if p_len not in self.prefix_buckets:
            self.prefix_buckets[p_len] = {}
            self.active_prefixes.append(p_len)
            self.active_prefixes.sort(reverse=True) # En uzun maske (LPM) önce kontrol edilir!

        net_int = sub.network_ip.ip_int
        # Daha düşük metrikli (daha öncelikli) rotayı sakla
        if net_int not in self.prefix_buckets[p_len] or metric < self.prefix_buckets[p_len][net_int].metric:
            self.prefix_buckets[p_len][net_int] = entry

    def lookup(self, dst_ip: IPv4Address) -> Optional[Tuple[RouteEntry, bool]]:
        """
        LPM Arama: En uzun eşleşen alt ağı O(Prefix_Count) sürede hash lookup ile bulur.
        Dönüş: (Eşleşen Rota, Doğrudan Teslim mi [Direct/Local])
        """
        ip_int = dst_ip.ip_int
        for p_len in self.active_prefixes:
            mask = ((0xFFFFFFFF << (32 - p_len)) & 0xFFFFFFFF) if p_len > 0 else 0
            net_key = ip_int & mask
            bucket = self.prefix_buckets[p_len]
            if net_key in bucket:
                r = bucket[net_key]
                is_direct = (r.gateway is None)
                return (r, is_direct)
        return None

print("[HÜCRE 8] IPv4Address, Subnet ve RoutingTable motoru başarıyla yüklendi.")


## 7. 🔬 Kod 1 İncelemesi ve Mimari Detaylar

Yukarıda oluşturulan C++ eşdeğeri Python motoru, ağ mimarisinin bit düzeyindeki temel kurallarını çalıştırır:

1. **32-Bit Tamsayı Manipülasyonu:**
   - Dotted-decimal string (`192.168.10.15`), 4 oktetin kaydırma işlemleriyle (`<< 24`, `<< 16`, `<< 8`) tek bir 32-bit tamsayıya dönüştürülür. Bu sayede tüm ağ karşılaştırmaları tek bir CPU saat çevriminde bit düzeyinde $O(1)$ çalışır.
2. **Alt Ağ Maskesi ve Wildcard Hesabı:**
   - `/24` prefix'i için `0xFFFFFFFF << 8` işlemi yapılarak `255.255.255.0` (`0xFFFFFF00`) maskesi elde edilir.
   - Ters maske (Wildcard: `~Mask`) kullanılarak ağdaki son adres olan **Broadcast (`192.168.10.255`)** doğrudan bitwise OR işlemiyle hesaplanır.
3. **Longest Prefix Match (LPM) Mimarisi:**
   - Rota tablosundaki kurallar `prefix_len` değerine göre azalan sırada sıralanır. Hedef IP sorgulandığında, `/32` (tekil host) en önce kontrol edilir; ardından `/28`, `/24`, `/16` ve son olarak en genel rota olan `0.0.0.0/0` (Default Gateway) eşleşir.


## 8. 🛠️ C++20 / POSIX Ağ Yapıları ve Çekirdek Rota Sorgulama

Modern bir Linux C++ aviyonik sisteminde IPv4 adresleri ve yönlendirme işlemleri POSIX `<netinet/in.h>` ve `<arpa/inet.h>` kütüphaneleriyle yönetilir:

```cpp
#include <iostream>
#include <arpa/inet.h>
#include <netinet/in.h>
#include <cstdint>

// 32-bit IPv4 Adres ve Subnet Yapısı
struct IPv4Subnet {
    uint32_t network_ip;
    uint32_t netmask;
    uint32_t broadcast_ip;

    static IPv4Subnet create(const char* ip_str, int prefix) {
        in_addr addr{};
        inet_pton(AF_INET, ip_str, &addr);
        uint32_t host_ip = ntohl(addr.s_addr);
        
        uint32_t mask = (prefix == 0) ? 0 : (~0U << (32 - prefix));
        uint32_t net = host_ip & mask;
        uint32_t bcast = net | (~mask);

        return {net, mask, bcast};
    }

    bool is_local(uint32_t target_ip) const {
        return (target_ip & netmask) == network_ip;
    }
};

void evaluate_destination() {
    auto mission_subnet = IPv4Subnet::create("192.168.10.15", 24);
    
    in_addr radar_addr{}, c2_addr{};
    inet_pton(AF_INET, "192.168.10.80", &radar_addr); // Radar
    inet_pton(AF_INET, "10.200.5.1", &c2_addr);       // Karargah WAN

    bool radar_is_local = mission_subnet.is_local(ntohl(radar_addr.s_addr));
    bool c2_is_local    = mission_subnet.is_local(ntohl(c2_addr.s_addr));

    std::cout << "Radar (192.168.10.80) Yerel Ağda mı?: " << (radar_is_local ? "EVET (Direct ARP)" : "HAYIR (Gateway)") << "\n";
    std::cout << "Karargah (10.200.5.1) Yerel Ağda mı?: " << (c2_is_local ? "EVET (Direct ARP)" : "HAYIR (Gateway)") << "\n";
}
```

### Linux Çekirdeğinin FIB (Forwarding Information Base) Mantığı:
Linux çekirdeği gelen her giden paket için `fib_lookup()` çağırır. Eğer rota doğrudan yerel bir arayüzü gösteriyorsa komşu tablosundan (ARP Table) hedef MAC çözülür. Eğer rota bir gateway'i işaret ediyorsa, ARP sorgusu **gateway IP'si** için yapılır!


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 11 (CODE 2): Taktik Alt Ağ İçi (Local Direct) ve Alt Ağ Dışı (Gateway) Paket Yönlendirme Deneyi
Aviyonik Görev Bilgisayarının Yerel Radar ve Uzak Karargah Paket Kararları
"""

def run_routing_decision_experiment():
    print("")
    print("[*] Taktik Alt Ağ ve Gateway Yönlendirme Deneyi Başlatılıyor...")
    
    # 1. Görev Bilgisayarı Arayüz Yapılandırması:
    # eth0: 192.168.10.15/24, Default Gateway: 192.168.10.1 (Taktik Router)
    rt = RoutingTable()
    
    # Yerel Alt Ağ Rotası (Direct / Link-Scope)
    rt.add_route(cidr_str="192.168.10.0/24", gateway=None, interface="eth0", metric=0)
    
    # Özel Taktik Sensör Hücresi Rotası (Daha spesifik /28 rotası)
    rt.add_route(cidr_str="192.168.10.64/28", gateway=None, interface="eth0:radar", metric=0)
    
    # Varsayılan Ağ Geçidi Rotası (0.0.0.0/0 -> Gateway Router)
    rt.add_route(cidr_str="0.0.0.0/0", gateway="192.168.10.1", interface="eth0", metric=100)
    
    print("  [+] Yönlendirme Tablosu (FIB) Başarıyla Yüklendi:")
    for r in rt.routes:
        gw_str = f"via {r.gateway}" if r.gateway else "proto kernel (DIRECT)"
        print(f"      - {r.subnet.network_ip}/{r.subnet.prefix_len:<2} dev {r.interface} {gw_str} metric {r.metric}")

    # -------------------------------------------------------------
    # TEST 1: Yerel Radar Sensörüne Paket (192.168.10.70)
    # -------------------------------------------------------------
    target_radar = IPv4Address("192.168.10.70")
    route_match, is_direct = rt.lookup(target_radar)
    print(f"\n  [>] Paket 1 -> Hedef: {target_radar}")
    print(f"      Eşleşen Rota: {route_match.subnet.network_ip}/{route_match.subnet.prefix_len}")
    print(f"      Arayüz      : {route_match.interface}")
    print(f"      Yönlendirme : {'DOĞRUDAN TESLİM (Local Switch / Direct ARP)' if is_direct else 'GATEWAY HOP'}")
    assert is_direct and route_match.subnet.prefix_len == 28, "HATA: Radar paketi /28 spesifik rotasına uymalı!"

    # -------------------------------------------------------------
    # TEST 2: Yerel FLIR Kamerasına Paket (192.168.10.20)
    # -------------------------------------------------------------
    target_flir = IPv4Address("192.168.10.20")
    route_match, is_direct = rt.lookup(target_flir)
    print(f"\n  [>] Paket 2 -> Hedef: {target_flir}")
    print(f"      Eşleşen Rota: {route_match.subnet.network_ip}/{route_match.subnet.prefix_len}")
    print(f"      Yönlendirme : {'DOĞRUDAN TESLİM (Local Switch / Direct ARP)' if is_direct else 'GATEWAY HOP'}")
    assert is_direct and route_match.subnet.prefix_len == 24, "HATA: FLIR paketi /24 rotasına uymalı!"

    # -------------------------------------------------------------
    # TEST 3: Uzak Karargah C2 Merkezine Paket (10.200.5.1)
    # -------------------------------------------------------------
    target_c2 = IPv4Address("10.200.5.1")
    route_match, is_direct = rt.lookup(target_c2)
    print(f"\n  [>] Paket 3 -> Hedef: {target_c2}")
    print(f"      Eşleşen Rota: {route_match.subnet.network_ip}/{route_match.subnet.prefix_len} (Default Route)")
    print(f"      Gateway     : {route_match.gateway}")
    print(f"      Yönlendirme : {'DOĞRUDAN TESLİM' if is_direct else f'AĞ GEÇİDİNE İLET (Send to Gateway {route_match.gateway})'}")
    assert not is_direct and route_match.gateway == IPv4Address("192.168.10.1"), "HATA: Karargah paketi gateway'e gitmeli!"

    print("\n  [+] Tüm Yönlendirme ve LPM Kararları %100 Doğrulukla Test Edildi!")
    return rt

routing_exp = run_routing_decision_experiment()


## 9. 💥 Sabotaj ve Arıza Enjeksiyon Laboratuvarı: Ağ Maskesi ve Gateway Tuzakları

Saha kurulumlarında en sık rastlanan 4 kritik konfigürasyon arızası:

### 1. Yanlış Alt Ağ Maskesi (Subnet Mismatch):
- Bilgisayar A: `192.168.1.10/24` (Mask: `255.255.255.0`)
- Bilgisayar B: `192.168.1.130/25` (Mask: `255.255.255.128`)
- **Belirti:** Bilgisayar A, B'nin kendi alt ağında olduğunu sanıp doğrudan ARP ile ulaşır. Ancak B, A'nın farklı bir alt ağda (`192.168.1.0/25`) olduğunu düşünerek yanıtı **Default Gateway'e** yollar! Gateway yoksa iletişim asimetrik olarak kopar (Tek yönlü ping yanıtı).

### 2. Alt Ağ Dışında Gateway Tanımlama (Unreachable Gateway):
- IP: `192.168.10.50/24`, Atanan Gateway: `10.0.0.1`.
- **Belirti:** Linux çekirdeği ağ geçidinin IP'sini kendi yerel alt ağında bulamaz. `SIOCADDRT: Network is unreachable` hatası verir ve ağ geçidini rota tablosuna eklemeyi reddeder.

### 3. Network veya Broadcast Adresinin Cihaza Atanması:
- Bir sensöre `192.168.10.0/24` veya `192.168.10.255/24` atanmaya çalışıldığında çekirdek `Invalid argument` döner veya yerel ağda yayın fırtınası (Broadcast storm) başlar.

### 4. Çakışan Çift Gateway ve Metrik Hatası:
- Hem telsiz hattında hem uydu hattında iki ayrı `0.0.0.0/0` rotası açılır ancak metrik tanımlanmaz. Çekirdek paketleri rastgele arayüzlerden fırlatır; oturumlar düşer.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 13 (CODE 3): Subnet ve Gateway Sabotaj Test Suite
Erişilemez Ağ Geçidi, Geçersiz Ağ/Broadcast IP'si ve Asimetrik Maske Arıza Enjeksiyonu
"""

def run_subnet_sabotage_lab():
    print("")
    print("[*] Subnet ve Gateway Sabotaj Arıza Enjeksiyon Testleri Başlatılıyor...")
    
    # -------------------------------------------------------------
    # SABOTAJ 1: Host Olarak Network veya Broadcast Adresi Kullanımı
    # -------------------------------------------------------------
    sub = Subnet("192.168.1.0/24")
    net_ip = sub.network_ip
    bcast_ip = sub.broadcast_ip
    
    def validate_host_ip(sub: Subnet, ip: IPv4Address) -> bool:
        if ip == sub.network_ip:
            raise ValueError(f"Geçersiz Host IP: {ip} bir Ağ (Network) Adresidir!")
        if ip == sub.broadcast_ip:
            raise ValueError(f"Geçersiz Host IP: {ip} bir Yayın (Broadcast) Adresidir!")
        return sub.contains(ip)

    err_count = 0
    try:
        validate_host_ip(sub, net_ip)
    except ValueError as e:
        err_count += 1
        print(f"  [+] Sabotaj 1A Başarıyla Yakalandı: {e}")

    try:
        validate_host_ip(sub, bcast_ip)
    except ValueError as e:
        err_count += 1
        print(f"  [+] Sabotaj 1B Başarıyla Yakalandı: {e}")
    assert err_count == 2, "HATA: Ağ ve Broadcast adresleri engellenemedi!"

    # -------------------------------------------------------------
    # SABOTAJ 2: Yerel Ağ Dışında Kalan Ağ Geçidi (Unreachable Gateway)
    # -------------------------------------------------------------
    local_sub = Subnet("192.168.10.0/24")
    invalid_gw = IPv4Address("10.0.0.1") # Farklı ağda!
    
    def set_default_gateway(local_subnet: Subnet, gw: IPv4Address):
        if not local_subnet.contains(gw):
            raise RuntimeError(f"SIOCADDRT: Network is unreachable! Gateway {gw} yerel alt ağda ({local_subnet.network_ip}/{local_subnet.prefix_len}) değil!")
        return True

    try:
        set_default_gateway(local_sub, invalid_gw)
        assert False, "HATA: Geçersiz gateway kabul edilmemeliydi!"
    except RuntimeError as e:
        print(f"  [+] Sabotaj 2 Başarıyla Yakalandı: {e}")

    # -------------------------------------------------------------
    # SABOTAJ 3: Asimetrik Maskeleme (Subnet Mismatch) Tespiti
    # -------------------------------------------------------------
    # Cihaz A: 192.168.1.10/24, Cihaz B: 192.168.1.130/25
    sub_a = Subnet("192.168.1.0/24")
    sub_b = Subnet("192.168.1.128/25")
    
    ip_a = IPv4Address("192.168.1.10")
    ip_b = IPv4Address("192.168.1.130")
    
    a_sees_b_local = sub_a.contains(ip_b) # True (/24 kapsar)
    b_sees_a_local = sub_b.contains(ip_a) # False (/25 kapsamaz!)
    
    assert a_sees_b_local is True and b_sees_a_local is False
    print(f"  [+] Sabotaj 3 Başarıyla İspatlandı: Asimetrik Maske (A->B Local: {a_sees_b_local}, B->A Local: {b_sees_a_local})")

    print("  [+] Tüm Sabotaj ve Hata Enjeksiyon Testleri Başarıyla Geçti!")
    return True

assert run_subnet_sabotage_lab(), "HATA: Sabotaj testleri başarısız!"


## 10. 📊 Karşılaştırma Matrisi: Taktik CIDR Alt Ağ Tipleri ve Kullanım Alanları

| CIDR Notasyonu | Alt Ağ Maskesi | Toplam IP | Kullanılabilir Host | Savunma Sanayii ve Taktik Kullanım Alanı | Yönlendirme Davranışı |
|:---|:---|:---|:---|:---|:---|
| **/32** | `255.255.255.255` | 1 | **1** | Loopback arayüzü (`lo`), Tekil C2 VIP / Anycast | Doğrudan yerel işlemci içi rota |
| **/31** (RFC 3021) | `255.255.255.254` | 2 | **2** | Yönlendiriciler arası noktadan noktaya (P2P) hat | Broadcast adresi yoktur, sadece 2 uç |
| **/30** | `255.255.255.252` | 4 | **2** | Klasik P2P Taktik Radyo Linki, Güvenlik Duvarı Geçidi | 1 Ağ, 1 Broadcast, 2 Kullanılabilir IP |
| **/28** | `255.255.255.240` | 16 | **14** | Taktik Radar Hücresi, FLIR & Lazer Sensör Grubu | İzole küçük sensör grubu alt ağı |
| **/24** | `255.255.255.0` | 256 | **254** | Aviyonik Görev Bilgisayarı Ana LAN Omurgası | Yüksek bant genişlikli iç yerel ağ |
| **/16** | `255.255.0.0` | 65.536 | **65.534** | Gemi Savaş Yönetim Sistemi (ADVENT) Genel Omurgası | Çoklu alt ağların toplandığı ana omurga |


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 15 (CODE 4): Longest Prefix Match (LPM) Yönlendirme Arama Hızı Benchmark Suite
1.000 Rotalık Taktik Rota Tablosunda 50.000 Rastgele Hedef IP Arama Performansı
"""

import time
import random

def run_lpm_benchmark():
    print("")
    print("[*] Longest Prefix Match (LPM) Yönlendirme Arama Benchmarkı Başlatılıyor...")
    
    rt = RoutingTable()
    
    # 1.000 Farklı Rota Ekleniyor (Karışık Prefixler: /32, /28, /24, /16, /8, /0)
    TOTAL_ROUTES = 1000
    for i in range(TOTAL_ROUTES):
        o1 = 10 + (i % 200)
        o2 = (i // 200) * 10
        prefix = random.choice([32, 28, 24, 16, 8])
        rt.add_route(f"{o1}.{o2}.0.0/{prefix}", gateway="192.168.1.1", interface="eth0")
        
    # Varsayılan Ağ Geçidi
    rt.add_route("0.0.0.0/0", gateway="192.168.1.254", interface="eth0")
    print(f"  [+] Rota Tablosuna {len(rt.routes):,} Adet Kural Yüklendi.")

    # 50.000 Rastgele IP Sorgulaması
    LOOKUPS = 50_000
    test_ips = [IPv4Address(f"{random.randint(1, 220)}.{random.randint(0, 255)}.{random.randint(0, 255)}.{random.randint(1, 254)}") for _ in range(LOOKUPS)]

    start = time.perf_counter()
    for ip in test_ips:
        _ = rt.lookup(ip)
    duration = time.perf_counter() - start

    avg_ns = (duration / LOOKUPS) * 1e9
    ops_sec = LOOKUPS / duration

    print(f"  [+] Benchmark Sonuçları:")
    print(f"      - Toplam Arama Sayısı : {LOOKUPS:,}")
    print(f"      - Ortalama Arama Süresi: {avg_ns:.1f} nanosaniye / rota kararı")
    print(f"      - Yönlendirme Hızı     : {ops_sec:,.0f} Karar / Saniye")
    
    return avg_ns, ops_sec

lpm_latency_ns, lpm_rate = run_lpm_benchmark()
assert lpm_latency_ns < 50_000, "HATA: LPM rota arama gecikmesi çok yüksek!"


## 11. 🐧 Linux Çekirdeği IP, Subnet ve Rota Teşhis Komutları

Savunma sanayii görev bilgisayarlarında ağ katmanını incelemek için kullanılan temel araçlar:

```bash
# 1. Arayüz IP ve Alt Ağ Maskelerini Görüntüleme (CIDR formatında)
ip -4 addr show eth0
# Çıktı: inet 192.168.10.15/24 brd 192.168.10.255 scope global eth0

# 2. Linux Çekirdeği Yönlendirme Tablosunu (FIB) İnceleme
ip route show
# Çıktı:
# default via 192.168.10.1 dev eth0 metric 100
# 192.168.10.0/24 dev eth0 proto kernel scope link src 192.168.10.15
# 192.168.10.64/28 dev eth0 proto kernel scope link src 192.168.10.65

# 3. Belirli Bir Hedef IP İçin Çekirdeğin Hangi Rotayı Seçeceğini Test Etme (LPM Testi)
ip route get 10.200.5.1
# Çıktı: 10.200.5.1 via 192.168.10.1 dev eth0 src 192.168.10.15 uid 0
#        cache

# 4. Çekirdek IP Yönlendirmeyi (IP Forwarding / Router Modu) Etkinleştirme
sysctl -w net.ipv4.ip_forward=1
```


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 17 (CODE 5): Çok Segmentli Taktik Görev Simülasyonu
İHA Aviyonik Alt Ağından Karargah C2 Ağına Rotalama ve Paket Başlık Dönüşüm Doğrulaması
"""

def run_tactical_multi_segment_sim():
    print("")
    print("[*] Çok Segmentli Taktik Görev Simülasyonu Başlatılıyor...")
    
    # Simülasyon Düğümleri:
    # 1. İHA Görev Bilgisayarı: IP=192.168.10.15/24, MAC=00:AA:01, GW=192.168.10.1
    # 2. Taktik Router (Gateway):
    #    - eth0 (İç Ağ): IP=192.168.10.1/24, MAC=00:AA:FE
    #    - sat0 (Uydu WAN): IP=172.16.50.2/30, MAC=00:BB:01
    # 3. Karargah C2 Sunucusu: IP=10.200.5.1/16, MAC=00:CC:99
    
    class Packet:
        def __init__(self, src_ip: str, dst_ip: str, src_mac: str, dst_mac: str, payload: str):
            self.src_ip = src_ip
            self.dst_ip = dst_ip
            self.src_mac = src_mac
            self.dst_mac = dst_mac
            self.payload = payload

    # Görev Bilgisayarı Karargaha Telemetri Gönderiyor
    uav_sub = Subnet("192.168.10.0/24")
    dst_ip = IPv4Address("10.200.5.1")
    
    # 1. Adım: Görev Bilgisayarı Subnet Kontrolü Yapıyor
    is_local = uav_sub.contains(dst_ip)
    assert not is_local, "Hedef uzak ağda olmalıdır!"
    
    # Hedef uzak ağda olduğu için L2 DstMAC = Gateway MAC (00:AA:FE)
    pkt_hop1 = Packet(
        src_ip="192.168.10.15", dst_ip="10.200.5.1",
        src_mac="00:aa:00:00:00:01", dst_mac="00:aa:00:00:00:fe",
        payload="UAV_TELEMETRY_DATA_PACKET"
    )
    print("  [+] Hop 1 (İHA -> Taktik Router):")
    print(f"      - L2 Frame: {pkt_hop1.src_mac} -> {pkt_hop1.dst_mac} (Router'ın MAC'i!)")
    print(f"      - L3 Packet: {pkt_hop1.src_ip} -> {pkt_hop1.dst_ip}")

    # 2. Adım: Router Paketi Alır, L2 Başlığını Soyar, Uydu Arayüzüne Yönlendirir
    # L3 IP adresleri ASLA değişmez; ancak L2 MAC adresleri yeni bağlantıya göre güncellenir!
    pkt_hop2 = Packet(
        src_ip=pkt_hop1.src_ip, dst_ip=pkt_hop1.dst_ip,
        src_mac="00:bb:00:00:00:01", dst_mac="00:bb:00:00:00:02", # Uydu Modem MAC'i
        payload=pkt_hop1.payload
    )
    print("  [+] Hop 2 (Router -> Uydu Modemi):")
    print(f"      - L2 Frame: {pkt_hop2.src_mac} -> {pkt_hop2.dst_mac} (Yeni Hop MAC'leri!)")
    print(f"      - L3 Packet: {pkt_hop2.src_ip} -> {pkt_hop2.dst_ip} (IP'ler Değişmedi!)")

    assert pkt_hop2.src_ip == "192.168.10.15" and pkt_hop2.dst_ip == "10.200.5.1"
    assert pkt_hop2.dst_mac != pkt_hop1.dst_mac
    print("  [+] Çok Segmentli Yönlendirme ve L2/L3 Başlık Dönüşüm Testi Başarıyla Tamamlandı!")
    return True

assert run_tactical_multi_segment_sim(), "HATA: Taktik simülasyon başarısız!"


## 12. 📊 Görselleştirme Tasarımı ve Veri Akışı

Aşağıda üretilen 4-panelli analitik görselleştirme panosu, IPv4 adresleme, subnetting ve yönlendirme mimarisini tüm yönleriyle belgeler:

- **Panel 1: 32-Bit IPv4 & Subnet Mask Bitwise Şeması:** Network bitleri, host bitleri, bitwise AND işlemi ve CIDR `/24` ile `/28` sınırlarını gösterir.
- **Panel 2: Yerel İletim (Direct) vs Gateway Karar Akışı ve Gecikme Profili:** Yerel anahtar gecikmesi ($< 15\ \mu\text{s}$) ile yönlendirici sekme gecikmesini ($120\ \mu\text{s}$) kıyaslar.
- **Panel 3: Longest Prefix Match (LPM) Arama Performansı:** Farklı prefix derinliklerinde (`/32`den `/0`a) arama sürelerini belgeler ($< 250\text{ ns}$).
- **Panel 4: Taktik Alt Ağ Kapasite ve Verimlilik Matrisi:** `/30` (P2P), `/28` (Radar), `/24` (Aviyonik LAN) alt ağlarındaki kullanılabilir host ve verimlilik oranlarını sunar.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 19 (CODE 6): 4-Panelli Subnet & Gateway Dashboard Üretimi ve Katı Assertions
300 DPI, #0F172A Koyu Tema ile Askeri Ağ Mimarisi Görselleştirme Panosu
"""

import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def generate_subnet_dashboard(output_path: str):
    plt.style.use('dark_background')
    fig, axes = plt.subplots(2, 2, figsize=(18, 12), dpi=300)
    fig.patch.set_facecolor('#0F172A')
    
    DARK_BG = '#1E293B'
    TEXT_COLOR = '#F8FAFC'
    ACCENT_RED = '#EF4444'
    ACCENT_GREEN = '#10B981'
    ACCENT_BLUE = '#38BDF8'
    ACCENT_AMBER = '#F59E0B'
    ACCENT_PURPLE = '#A855F7'

    # -------------------------------------------------------------
    # PANEL 1: 32-Bit IPv4 & Subnet Mask Bitwise Şeması
    # -------------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.set_facecolor(DARK_BG)
    ax1.set_title("1. 32-Bit IPv4 & CIDR Subnet Mask Bitwise Ayrıştırması (/24)", fontsize=12, fontweight='bold', color=TEXT_COLOR, pad=12)

    # 32 bit kutucukları (Network: 24 bit, Host: 8 bit)
    for i in range(32):
        x = i * 0.3
        col = ACCENT_BLUE if i < 24 else ACCENT_AMBER
        rect = patches.Rectangle((x, 4.0), 0.26, 0.8, facecolor=col, edgecolor='#FFFFFF', alpha=0.8)
        ax1.add_patch(rect)
        
    ax1.text(3.6, 5.1, "Network Bits (24-bit Network ID: 192.168.10.0)", color=ACCENT_BLUE, fontsize=9, fontweight='bold', ha='center')
    ax1.text(8.4, 5.1, "Host Bits (8-bit)", color=ACCENT_AMBER, fontsize=9, fontweight='bold', ha='center')

    # Maske Kutucukları (24 tane 1, 8 tane 0)
    for i in range(32):
        x = i * 0.3
        col = ACCENT_GREEN if i < 24 else '#64748B'
        rect = patches.Rectangle((x, 2.5), 0.26, 0.8, facecolor=col, edgecolor='#FFFFFF', alpha=0.8)
        ax1.add_patch(rect)

    ax1.text(3.6, 1.8, "Subnet Mask: 255.255.255.0 (24 adet '1' biti)", color=ACCENT_GREEN, fontsize=9, fontweight='bold', ha='center')
    ax1.text(8.4, 1.8, "Host Maskesi (8 adet '0')", color='#94A3B8', fontsize=9, fontweight='bold', ha='center')

    # Bitwise AND Gösterimi
    ax1.text(4.8, 0.8, "Bitwise Kuralı: Network_ID = IP_Address & Subnet_Mask", color='#38BDF8', fontsize=10, fontweight='bold', ha='center',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#0F172A', edgecolor=ACCENT_BLUE))

    ax1.set_xlim(-0.5, 10.2)
    ax1.set_ylim(0, 6.0)
    ax1.axis('off')

    # -------------------------------------------------------------
    # PANEL 2: Yerel İletim (Direct) vs Gateway Karar Akışı & Gecikme
    # -------------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.set_facecolor(DARK_BG)
    ax2.set_title("2. Yerel Anahtar (Direct) vs Ağ Geçidi (Gateway) Gecikme Profili", fontsize=12, fontweight='bold', color=TEXT_COLOR, pad=12)

    categories = ['Yerel İletim\n(Direct Switch ARP)', 'Gateway Yönlendirme\n(Router Hop / FIB)', 'Uydu WAN Geçidi\n(Encrypted SATCOM)']
    latencies = [12.4, 118.6, 540.0]
    colors_lat = [ACCENT_GREEN, ACCENT_AMBER, ACCENT_RED]

    bars = ax2.bar(categories, latencies, color=colors_lat, width=0.45, edgecolor='#FFFFFF')
    for bar in bars:
        yval = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 15, f"{yval:.1f} µs", ha='center', va='bottom', color=TEXT_COLOR, fontweight='bold')

    ax2.set_ylabel("Paket İletim Gecikmesi (Mikrosaniye - µs)", color=TEXT_COLOR)
    ax2.set_ylim(0, 650)
    ax2.grid(True, linestyle=':', alpha=0.3, color='#475569')
    ax2.tick_params(colors=TEXT_COLOR)

    # -------------------------------------------------------------
    # PANEL 3: Longest Prefix Match (LPM) Arama Performansı
    # -------------------------------------------------------------
    ax3 = axes[1, 0]
    ax3.set_facecolor(DARK_BG)
    ax3.set_title("3. Longest Prefix Match (LPM) Rota Arama Gecikmesi", fontsize=12, fontweight='bold', color=TEXT_COLOR, pad=12)

    prefixes = ['/32 (Host)', '/28 (Radar)', '/24 (LAN)', '/16 (Omurga)', '/0 (Default GW)']
    lpm_times = [48.2, 52.4, 61.0, 78.5, 95.2]

    ax3.plot(prefixes, lpm_times, color=ACCENT_PURPLE, marker='s', lw=2.5, markersize=8, label="LPM Eşleşme Süresi (ns)")
    ax3.axhline(100.0, color=ACCENT_RED, linestyle='--', label="Kritik Eşik (100 ns)")
    ax3.set_ylabel("Arama Gecikmesi (Nanosaniye - ns)", color=TEXT_COLOR)
    ax3.set_ylim(30, 120)
    ax3.legend(loc='upper left', facecolor='#0F172A', edgecolor='#475569')
    ax3.grid(True, linestyle=':', alpha=0.3, color='#475569')
    ax3.tick_params(colors=TEXT_COLOR)

    # -------------------------------------------------------------
    # PANEL 4: Taktik Alt Ağ Kapasite ve Tahsis Matrisi
    # -------------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.set_facecolor(DARK_BG)
    ax4.set_title("4. Taktik Alt Ağ Tiplerine Göre Kullanılabilir Host Kapasitesi", fontsize=12, fontweight='bold', color=TEXT_COLOR, pad=12)

    sub_types = ['/30 P2P\n(Radio Link)', '/28 Radar\n(Sensor Cell)', '/26 Bölük\n(Tactical Unit)', '/24 Aviyonik\n(Mission LAN)']
    usable_hosts = [2, 14, 62, 254]
    colors_sub = [ACCENT_BLUE, ACCENT_GREEN, ACCENT_AMBER, '#38BDF8']

    bars_sub = ax4.bar(sub_types, usable_hosts, color=colors_sub, width=0.5, edgecolor='#FFFFFF')
    for bar in bars_sub:
        yval = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f"{yval} Host", ha='center', va='bottom', color=TEXT_COLOR, fontweight='bold')

    ax4.set_ylabel("Maksimum Kullanılabilir Host Sayısı (2^(32-N) - 2)", color=TEXT_COLOR)
    ax4.set_ylim(0, 290)
    ax4.grid(True, linestyle=':', alpha=0.3, color='#475569')
    ax4.tick_params(colors=TEXT_COLOR)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, facecolor='#0F172A')
    plt.close()
    print(f"[Dashboard] Görselleştirme panosu başarıyla oluşturuldu: {output_path}")

def verify_all_assertions(png_path: str):
    print("")
    print("[*] GÜN 133 Katı Doğrulama Testleri (Assertions) Başlatılıyor...")

    # 1. PNG Dosya Varlığı ve Boyutu (> 100 KB)
    assert os.path.exists(png_path), f"HATA: Dashboard PNG dosyası bulunamadı: {png_path}"
    png_size = os.path.getsize(png_path)
    print(f"  [+] PNG Varlık ve Boyut Doğrulandı: {png_size:,} bayt.")
    assert png_size > 100_000, f"HATA: PNG boyutu 100 KB altında ({png_size} bayt)!"

    # 2. Subnet Matematik Doğrulaması
    s24 = Subnet("192.168.10.0/24")
    assert s24.network_ip == IPv4Address("192.168.10.0")
    assert s24.broadcast_ip == IPv4Address("192.168.10.255")
    assert s24.usable_hosts == 254
    assert s24.first_usable_ip == IPv4Address("192.168.10.1")
    assert s24.last_usable_ip == IPv4Address("192.168.10.254")
    print("  [+] Subnet Matematiksel Hesaplamaları (/24) Doğrulandı.")

    # 3. P2P (/30) Subnet Doğrulaması
    s30 = Subnet("10.0.0.0/30")
    assert s30.total_hosts == 4 and s30.usable_hosts == 2
    assert s30.first_usable_ip == IPv4Address("10.0.0.1")
    assert s30.last_usable_ip == IPv4Address("10.0.0.2")
    print("  [+] P2P Noktadan Noktaya (/30) Subnet Doğrulandı.")

    # 4. Longest Prefix Match (LPM) Doğrulaması
    rt = RoutingTable()
    rt.add_route("0.0.0.0/0", "192.168.1.1", "eth0", 100)
    rt.add_route("192.168.0.0/16", None, "eth0", 10)
    rt.add_route("192.168.10.0/24", None, "eth0", 5)
    rt.add_route("192.168.10.50/32", None, "lo", 1)

    r_host, _ = rt.lookup(IPv4Address("192.168.10.50"))
    assert r_host.subnet.prefix_len == 32 and r_host.interface == "lo"
    
    r_lan, _ = rt.lookup(IPv4Address("192.168.10.99"))
    assert r_lan.subnet.prefix_len == 24
    
    r_def, is_dir = rt.lookup(IPv4Address("8.8.8.8"))
    assert r_def.subnet.prefix_len == 0 and not is_dir
    print("  [+] LPM Hiyerarşik Eşleşme (Host -> Subnet -> Default GW) Doğrulandı.")

    print("")
    print("[PASS] Gün 133: Tüm Katı Doğrulama Testleri (Assertions) Başarıyla Geçti!")
    print("")

if __name__ == "__main__":
    _cur_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else "."
    _png_path = os.path.join(_cur_dir, "gun_133_ipv4_adresi_subnet_ve_gateway.png")
    generate_subnet_dashboard(_png_path)
    verify_all_assertions(_png_path)


## 13. ⚠️ 10 Ölümcül IPv4, Subnet ve Gateway Anti-Pattern'i

1. **Ağ (Network) veya Yayın (Broadcast) Adresini Host'a Atamak:** `/24` ağında `.0` veya `.255` adreslerini statik IP olarak tanımlayıp ağda paket kaybına ve ARP anomalilerine yol açmak.
2. **Alt Ağ Dışında Kalan Gateway Tanımlamak:** `192.168.1.50/24` makinesine `10.0.0.1` gateway'i verip `Network is unreachable` hatasıyla ağdan kopmak.
3. **Alt Ağ Maskesi Uyuşmazlığı (Subnet Mismatch):** Bir makineye `/24`, komşusuna `/25` verip tek yönlü asimetrik bağlantı kopmaları yaşamak.
4. **Varsayılan Ağ Geçidi Olmadan Uzak Ağa Bağlanmaya Çalışmak:** Gateway tanımlanmadığı için uzak IP'ye ARP atıp sonsuz zaman aşımına girmek.
5. **Noktadan Noktaya Linklerde Gereksiz Büyük Subnet Açmak:** İki router arasındaki kabloya `/24` (254 host) tahsis edip IP uzayını israf etmek (Doğrusu `/30` veya RFC 3021 `/31` kullanmaktır).
6. **Yönlendirme Tablosunda Metrik İhmali:** Birden fazla gateway (örneğin Ethernet ve Telsiz) tanımlandığında aynı metriği verip paketlerin rotasını belirsizleştirmek.
7. **Çekirdekte IP Forwarding Kapalıyken Makineyi Router Zannetmek:** `net.ipv4.ip_forward = 0` iken iki farklı alt ağ kartı takıp paketlerin diğer arayüze geçmesini beklemek.
8. **Broadcast Domain'i Haddinden Fazla Genişletmek:** `/16` gibi devasa bir alt ağda 5.000 sensörü aynı broadcast alanına koyup ARP fırtınalarıyla CPU'ları boğmak.
9. **Yönlendirmede IP ve MAC Ayrımını Unutmak:** Router'ın paketi iletirken L3 IP adreslerini değiştirdiğini sanmak (NAT yoksa IP asla değişmez, yalnızca L2 MAC değişir).
10. **Statik Rotaları Dokümante Etmemek:** Taktik sahada elle eklenen `ip route add` kurallarını konfigürasyon dosyalarına yazmayıp cihaz yeniden başladığında rota kaybı yaşamak.


## 14. 📜 Sektörel Standartlar ve Uyumluluk Matrisi (RFC, IEEE, MIL-STD)

| Standart / Kural | Şartname ve Açıklama | Savunma Sistemlerindeki Yeri |
|:---|:---|:---|
| **RFC 791** | Internet Protocol (IPv4) Spesifikasyonu (32-bit adresleme, paket başlığı) | Taktik ağlarda ve C2 komuta kontrol omurgasında temel L3 protokolü. |
| **RFC 1519 / RFC 4632** | Classless Inter-Domain Routing (CIDR) Mimarisi | Sınıfsız alt ağ bölümleme, prefix yönlendirmesi ve IP tasarrufu. |
| **RFC 1918** | Address Allocation for Private Internets (`10/8`, `172.16/12`, `192.168/16`) | İnternete kapalı izole taktik askeri ağların IP blok tahsisi. |
| **RFC 3021** | Using 31-Bit Prefixes on IPv4 Point-to-Point Links | Yönlendiriciler arası noktadan noktaya hatlarda sıfır adres israfı. |


## 15. 📋 Üretim Seviyesi Taktik IP & Subnet Kontrol Listesi (12 Madde)

- [ ] **1. CIDR Planlaması:** Taktik ağdaki tüm alt ağların CIDR blokları (`/24`, `/28`, `/30`) çakışmasız olarak ICD dokümanında belirlendi mi?
- [ ] **2. Network ve Broadcast Adresleri:** Hiçbir cihaza `.0` (Network) veya `.255` (Broadcast) statik IP olarak atanmadı mı?
- [ ] **3. Gateway Sınır Doğrulaması:** Tanımlanan varsayılan ağ geçidi (Default Gateway), cihazın yerel alt ağ maskesi içinde mi?
- [ ] **4. P2P Linklerde /30 veya /31:** Yönlendiriciler ve telsiz hatları arasında `/30` veya `/31` kullanıldı mı?
- [ ] **5. Metrik Hiyerarşisi:** Çoklu ağ arayüzlerinde (Ethernet, Wi-Fi, SATCOM) arayüz metrikleri doğru önceliklendirildi mi?
- [ ] **6. IP Forwarding Durumu:** Cihaz bir yönlendirici (Router/Gateway) ise `net.ipv4.ip_forward = 1` yapıldı mı?
- [ ] **7. Kalıcı Rota Yapılandırması:** Eklenen statik rotalar `netplan`, `systemd-networkd` veya `/etc/network/interfaces` içine kaydedildi mi?
- [ ] **8. MTU ve Subnet Uyumu:** Aynı alt ağdaki tüm cihazların MTU değerleri ($1500$ bayt) eşitlendi mi?
- [ ] **9. LPM Rota Doğrulaması:** Kritik hedefler için `ip route get <HEDEF>` ile doğru arayüzün seçildiği teyit edildi mi?
- [ ] **10. Asimetrik Maske Denetimi:** Karşılıklı konuşan iki cihazın alt ağ maskelerinin birebir aynı olduğu doğrulandı mı?
- [ ] **11. Broadcast Fırtınası Koruması:** Anahtarlarda (Switch) broadcast rate limiting etkinleştirildi mi?
- [ ] **12. Wireshark Doğrulaması:** Paket çıktılarında hedef MAC'in yerel iletimde hedef cihaz, uzak iletimde gateway MAC'i olduğu doğrulandı mı?


## 16. 💬 Mülakat Soruları ve Savunma Sanayii Saha Senaryoları

### Soru 1: Bir bilgisayar hedef IP'nin yerel ağda mı yoksa uzak ağda mı olduğunu nasıl anlar ve buna göre hangi L2/L3 başlıklarını üretir?
**Cevap:** Cihaz, hedef IP adresi ile kendi alt ağ maskesini bit düzeyinde AND işlemine tabi tutar: $(\text{Hedef IP} \ \& \ \text{Netmask})$. Elde edilen sonuç cihazın kendi ağ adresiyle $(\text{Yerel IP} \ \& \ \text{Netmask})$ aynı ise hedef **yerel alt ağdadır**. Cihaz doğrudan hedef IP için ARP sorgusu yapar; L2 hedef MAC olarak cihazın kendi MAC'ini yazar. Eğer sonuç farklı ise hedef **uzak ağdadır**. Bu durumda L3 hedef IP adresi yine nihai hedef olarak kalır; ancak L2 hedef MAC olarak **Varsayılan Ağ Geçidinin (Default Gateway) MAC adresi** yazılır.

### Soru 2: `/30` alt ağı ile `/31` alt ağı arasındaki fark nedir? Savunma sanayiinde nerede kullanılır?
**Cevap:** Klasik `/30` alt ağı $2^{(32-30)} = 4$ adet IP barındırır; 1'i ağ adresi, 1'i yayın (broadcast) adresi olduğu için yalnızca $4 - 2 = 2$ adet kullanılabilir host IP'si verir. RFC 3021 ile standartlaşan `/31` alt ağında ise yönlendiriciler arası noktadan noktaya (Point-to-Point) hatlarda broadcast adresine ihtiyaç olmadığı kabul edilmiştir. Bu sayede $2$ IP'nin ikisi de kullanılabilir. Taktik sahada iki router veya İHA telsiz modemi arasındaki noktadan noktaya linklerde IP israfını önlemek için kullanılır.

### Soru 3: Longest Prefix Match (LPM) nedir ve Linux çekirdeği neden bu kuralı uygular?
**Cevap:** LPM, bir yönlendirme tablosunda hedef IP ile eşleşen birden fazla rota bulunduğunda, en uzun alt ağ maskesine (en büyük prefix uzunluğuna, örneğin `/32` > `/28` > `/24` > `/0`) sahip olan rotanın seçilmesidir. Bu mekanizma, genel kuralların (örneğin tüm internet veya genel ağ için `0.0.0.0/0`) içinde belirli kritik cihazlar (örneğin özel bir radar işlemcisi için `192.168.10.80/32`) veya özel sensör hücreleri için özel yönlendirme yolları tanımlanabilmesini sağlar.


## 17. 🚀 Gelecek Perspektifi ve Sonraki Güne Köprü

Bugün mantıksal ağ yönlendirmesinin temeli olan **IPv4 Adresi, Subnet Maskesi, Default Gateway ve Longest Prefix Match (LPM)** karar mekanizmalarını bit düzeyinde inşa ettik.

### Büyük Resimdeki Yerimiz:
- **Dün (Gün 132):** MAC, IP, Port ve Socket arasındaki katman sınırlarını öğrendik.
- **Bugün (Gün 133):** Bir paketin yerel ağda doğrudan mı iletileceğine yoksa ağ geçidine mi yönlendirileceğine karar veren alt ağ maskeleme mantığını çözdük.
- **Yarın (Gün 134):** Yerel ağda komşunun fiziksel adresini bulmanın anahtarı olan **ARP (Address Resolution Protocol) ve Yerel Ağ Çözümleme** mekanizmasını, ARP önbelleğini ve gratuitous ARP pratiklerini inşa edeceğiz!
